In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import statsmodels.api as sm
import math
from scipy import stats
import matplotlib.pyplot as plt
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import plotly.io as pio
pio.renderers.default = "kaggle"

from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder


In [13]:
print("Current directory:", os.getcwd())
#should be rds_project

Current directory: /Users/georgebuck/Desktop/rds_project


In [15]:
train =  pd.read_csv(r"data/playground-series-s6e1/sample_submission.csv").drop(columns="id",axis=1,)
test = pd.read_csv(r"data/playground-series-s6e1/test.csv").drop(columns="id",axis=1,)
sub = pd.read_csv(r"data/playground-series-s6e1/sample_submission.csv")

In [24]:
train

,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,easy,78.300
1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,moderate,46.700
2,20,female,b.sc,4.68,92.6,yes,5.8,poor,coaching,high,moderate,99.000
3,19,male,b.sc,2.00,49.5,yes,8.3,average,group study,high,moderate,63.900
4,23,male,bca,7.65,86.9,yes,9.6,good,self-study,high,easy,100.000
...,...,...,...,...,...,...,...,...,...,...,...,...
629995,18,female,b.tech,4.86,70.7,yes,4.1,good,mixed,high,moderate,69.500
629996,21,female,ba,7.08,54.4,yes,4.5,average,mixed,low,moderate,78.900
629997,24,male,bca,0.64,44.2,yes,4.3,poor,online videos,low,moderate,19.599
629998,20,male,b.com,1.54,75.1,yes,8.2,average,group study,high,moderate,59.100


## Dataset Overview
- This dataset contains academic performance and lifestyle data for 630,000 students to analyze factors influencing exam success.

#### Column Descriptions
- id: A unique identifier assigned to each individual student record in the dataset.

- age: The chronological age of the student, ranging from 17 to 24 years.

- gender: The self-identified gender of the student (Male, Female, or Other).

- course: The specific academic program or degree the student is currently enrolled in.

- study_hours: The average number of hours the student spends studying on a daily basis.

- class_attendance: The percentage of scheduled classes the student has attended.

- internet_access: Indicates whether the student has a reliable internet connection for study (Yes/No).

- sleep_hours: The average duration of sleep the student gets per night.

- sleep_quality: A subjective rating of the student's sleep restorative value (Poor, Average, Good).

- study_method: The primary technique used for learning (e.g., Self-study, Coaching, Online videos).

- facility_rating: The student's assessment of their educational environment's quality (Low, Medium, High).

- exam_difficulty: The perceived or categorized level of challenge for the examination (Easy, Moderate, Hard).

- exam_score: The final numerical mark achieved by the student (The target variable for prediction).

In [16]:
data_des = train.select_dtypes(include='number').agg(['mean', 'std', 'median']).T

In [17]:
df_plot = data_des.reset_index()

In [27]:
fig = px.bar(
    df_plot, 
    x="index", 
    y=["mean", "median", "std"], 
    barmode="group", 
    log_y=True, 
    title="Comparison between Mean, Median and Standard Deviation",
    labels={"index": "Study Method", "value": "Score/Value", "variable": "Statistic"},
    template="plotly_dark"
)

fig.show(renderer="kaggle")

In [18]:
def cat_tar(df, target_col="exam_score"):
    cat_cols = df.select_dtypes("object").columns
    n_features = len(cat_cols)
    num_cols = math.ceil(n_features / 2)
    
    fig = make_subplots(
        rows=2, 
        cols=num_cols, 
        subplot_titles=[f"{c} vs {target_col}" for c in cat_cols]
    )

    for i, col_name in enumerate(cat_cols):
        curr_row = (i % 2) + 1
        curr_col = (i // 2) + 1
        
        data = df.groupby(col_name)[target_col].mean().reset_index().sort_values(by=target_col, ascending=False)
        
        fig.add_trace(
            go.Bar(
                x=data[col_name], 
                y=data[target_col], 
                name=col_name
            ),
            row=curr_row, 
            col=curr_col
        )

    fig.update_layout(
        height=900,
        title_text="Categorical Analysis vs Target", 
        showlegend=False
    )
    fig.show(renderer="kaggle")


In [19]:
cat_tar(train)

ValueError: 
The 'cols' argument to make_subplots must be an int greater than 0.
    Received value of type <class 'int'>: 0

In [20]:
def data_summary(df):
    num_col = df.select_dtypes("number").columns
    num_col = num_col[num_col != "exam_score"]
    X = df[num_col]
    y = df['exam_score']
    model = sm.OLS(y, X.astype(float)).fit()
    return model.summary()

    
    

In [21]:
data_summary(train)

ValueError: zero-size array to reduction operation maximum which has no identity

In [32]:
train.corr(numeric_only=True)*100

,age,study_hours,class_attendance,sleep_hours,exam_score
age,100.000000,0.754481,0.562825,0.586435,1.047241
study_hours,0.754481,100.000000,8.761716,4.249149,76.226733
class_attendance,0.562825,8.761716,100.000000,2.926309,36.095409
sleep_hours,0.586435,4.249149,2.926309,100.000000,16.740997
exam_score,1.047241,76.226733,36.095409,16.740997,100.000000


# T Test

In [33]:
internet_yes = train[train['internet_access'] == 'yes']['exam_score']
internet_no = train[train['internet_access'] == 'no']['exam_score']

t_stat, p_val = stats.ttest_ind(internet_yes, internet_no)

print(f"T-Statistic: {t_stat}, P-Value: {p_val}")

T-Statistic: 0.35496949770317443, P-Value: 0.7226125612563572


# One-way ANOVA

In [34]:
self_study = train[train['study_method'] == 'self-study']['exam_score']
coaching = train[train['study_method'] == 'coaching']['exam_score']
group_study = train[train['study_method'] == 'group study']['exam_score']

online_videos = train[train['study_method'] == 'online videos']['exam_score']
mixed = train[train['study_method'] == 'mixed']['exam_score']

f_stat, p_val = stats.f_oneway(self_study, coaching, group_study,online_videos ,mixed)

print(f"F-Statistic: {f_stat}, P-Value: {p_val}")

F-Statistic: 8304.288535811598, P-Value: 0.0


# Chi-Square Test 

In [35]:
contingency_table = pd.crosstab(train['gender'], train['exam_difficulty'])

chi2, p_val, dof, expected = stats.chi2_contingency(contingency_table)

print(f"Chi2 Statistic: {chi2}, P-Value: {p_val}")

Chi2 Statistic: 40.81660068053178, P-Value: 2.933391073825852e-08


In [36]:
tukey = pairwise_tukeyhsd(endog=train['exam_score'],      
                          groups=train['study_method'],   
                          alpha=0.05)                     

print(tukey)

        Multiple Comparison of Means - Tukey HSD, FWER=0.05        
    group1        group2    meandiff p-adj  lower    upper   reject
-------------------------------------------------------------------
     coaching   group study  -8.7348   0.0  -8.9342  -8.5354   True
     coaching         mixed  -4.1649   0.0  -4.3643  -3.9655   True
     coaching online videos  -9.5391   0.0  -9.7393  -9.3388   True
     coaching    self-study -11.5665   0.0 -11.7627 -11.3703   True
  group study         mixed   4.5699   0.0   4.3671   4.7727   True
  group study online videos  -0.8042   0.0  -1.0078  -0.6006   True
  group study    self-study  -2.8317   0.0  -3.0313  -2.6321   True
        mixed online videos  -5.3741   0.0  -5.5777  -5.1706   True
        mixed    self-study  -7.4016   0.0  -7.6012   -7.202   True
online videos    self-study  -2.0275   0.0  -2.2279   -1.827   True
-------------------------------------------------------------------


In [37]:
strange_students = train[(train['study_hours'] < 1) & (train['exam_score'] > 90)]
display(strange_students[['study_hours', 'exam_score', 'study_method']])

,study_hours,exam_score,study_method
51537,0.47,100.0,coaching
75952,0.08,93.5,group study
78392,0.86,100.0,online videos
80139,0.40,91.7,coaching
117555,0.08,93.4,coaching
194976,0.08,91.5,coaching
198833,0.28,97.5,coaching
258607,0.80,99.9,coaching
300877,0.90,91.6,coaching
307262,0.86,92.5,coaching


# model training 

In [38]:
le = LabelEncoder()
cat_cols = ['gender', 'course', 'internet_access', 'sleep_quality', 
            'study_method', 'facility_rating', 'exam_difficulty']

for col in cat_cols:
    train[col] = le.fit_transform(train[col])

for col in cat_cols:
    test[col] = le.fit_transform(test[col])

In [39]:
X = train.drop(columns=['exam_score'])
y = train['exam_score']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


model = HistGradientBoostingRegressor(
    max_iter=1000,        
    learning_rate=0.05,   
    max_depth=10,         
    random_state=42
)

# Model Train
print("Model Training Start...")
model.fit(X_train, y_train)

# Prediction 
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"\n--- Result ---")
print(f"R2 Score (Accurry): {r2:.4f}")
print(f"MAE (Mean Absoult Error): {mae:.4f} ")

def predict_new_student(new_data_row):
    prediction = model.predict(new_data_row)
    return prediction

print("Done")

Model Training Start...

--- Result ---
R2 Score (Accurry): 0.7836
MAE (Mean Absoult Error): 6.9982 
Done


In [40]:
sub['exam_score'] = predict_new_student(test)

In [42]:
sub.to_csv("students marks.csv",index=False)